In [1]:
%pip install pandas sqlalchemy pymysql

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 419.4 kB/s eta 0:00:04
   --------- ------------------------------ 0.5/2.2 MB 419.4 kB/s eta 0:00:04
   --------- ------------------------------ 0.5/2.2 MB 419.4 kB/s eta 0:00:04
   --------- ------------------------------ 0.5/2.2 MB 419.4 kB/s eta 0:00:04
   -------------- ------------------------- 0.8/2.2 MB 390.1 kB/s eta 0:00:04
   -------------- ------------------------- 0.8/2.2 MB 390.1 kB/s eta 0:00:04
   -------------- ------------------------- 0.8/2.2 MB 390.1 kB/s eta 0:00:04
   ------------------- -----------------

In [ ]:
#环境准备
from pathlib import Path
from getpass import getpass
import os

import pandas as pd
import numpy as np

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("模块导入完成。")

模块导入完成。


In [3]:
#测试MySQL连接
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_DATABASE = "ecommerce_analysis"

MYSQL_PASSWORD = os.getenv("242323225Wrh")
if not MYSQL_PASSWORD:
    MYSQL_PASSWORD = getpass("请输入 MySQL 密码（输入内容不会显示）：")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE,
    query={"charset": "utf8mb4"}
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

print("数据库 Engine 已创建。")

数据库 Engine 已创建。


In [4]:
#测试连接
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT DATABASE() AS current_database, VERSION() AS mysql_version")
    )
    connection_check = pd.DataFrame(result.fetchall(), columns=result.keys())

connection_check

,current_database,mysql_version
0,ecommerce_analysis,8.4.11


In [17]:
#检查dwd前置条件
required_columns = {
    "session_id",
    "Month",
    "Month_Num",
    "VisitorType",
    "VisitorType_Name",
    "TrafficType",
    "is_Weekend",
    "is_Purchase",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "TotalPage_Views",
    "TotalPage_Duration",
    "ProductView_Ratio"
}

columns_sql = text('''
SELECT COLUMN_NAME
FROM information_schema.COLUMNS
WHERE TABLE_SCHEMA = :db_name
  AND TABLE_NAME = 'dwd_user_session'
''')

with engine.connect() as conn:
    existing_columns = {
        row[0]
        for row in conn.execute(
            columns_sql,
            {"db_name": MYSQL_DATABASE}
        ).fetchall()
    }

missing_columns = required_columns - existing_columns

if missing_columns:
    raise RuntimeError(
        "dwd_user_session 缺少以下字段，请先检查第2课 DWD：\n"
        + "\n".join(sorted(missing_columns))
    )

dwd_check = pd.read_sql(
    '''
    SELECT
        COUNT(*) AS dwd_rows,
        SUM(is_purchase) AS purchase_sessions,
        ROUND(SUM(is_purchase) * 100.0 / COUNT(*), 2) AS conversion_rate
    FROM dwd_user_session
    ''',
    engine
)

dwd_check

,dwd_rows,purchase_sessions,conversion_rate
0,12330,1908.0,15.47


In [13]:
#定位独立的SQL文件
sql_candidates = [
    Path("../sql/06_dws_ads_etl.sql"),
    Path("sql/06_dws_ads_etl.sql"),
    Path("06_dws_ads_etl.sql"),
]

SQL_FILE = next(
    (p for p in sql_candidates if p.exists()),
    None
)

if SQL_FILE is None:
    raise FileNotFoundError(
        "没有找到 06_dws_ads_etl.sql。\n"
        "请确认项目结构为 python/04_dws_ads_etl.ipynb + sql/06_dws_ads_etl.sql。"
    )

print("找到 SQL 文件：", SQL_FILE.resolve())

找到 SQL 文件： F:\Ecommerce-User-Analysis\sql\06_dws_ads_etl.sql


In [18]:
#读取SQL脚本
sql_script = SQL_FILE.read_text(encoding="utf-8")

print(f"SQL 文件字符数：{len(sql_script):,}")
print(sql_script[:1200])

SQL 文件字符数：11,009
-- ============================================================
-- 第6课：DWS + ADS + SQL ETL
-- 项目：ecommerce-user-analysis
-- 前置依赖：ecommerce_analysis.dwd_user_session 已存在且已完成第2课清洗
-- 说明：本脚本可重复执行；每次会重建本课 DWS/ADS 表。
-- ============================================================

USE ecommerce_analysis;

-- 0. 源数据检查
SELECT
    COUNT(*) AS dwd_rows,
    SUM(is_Purchase) AS purchase_sessions,
    ROUND(SUM(is_Purchase) * 100.0 / COUNT(*), 2) AS conversion_rate
FROM dwd_user_session;

-- 1. DWS：月度经营汇总
DROP TABLE IF EXISTS dws_monthly_conversion;
CREATE TABLE dws_monthly_conversion AS
SELECT
    month_num,
    month,
    COUNT(*) AS session_cnt,
    SUM(is_Purchase) AS purchase_cnt,
    ROUND(SUM(is_Purchase) * 100.0 / COUNT(*), 2) AS conversion_rate,
    ROUND(AVG(ProductRelated), 2) AS avg_product_views,
    ROUND(AVG(ProductRelated_Duration), 2) AS avg_product_duration,
    ROUND(AVG(BounceRates), 4) AS avg_bounce_rate,
    ROUND(AVG(ExitRates), 4) AS avg_exit_rate,
    ROU

In [19]:
#执行DWS + ADS ETL
def remove_full_line_comments(script: str) -> str:
    cleaned_lines = []
    for line in script.splitlines():
        stripped = line.strip()
        if stripped.startswith("--") or stripped == "":
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)


def split_sql_statements(script: str):
    cleaned = remove_full_line_comments(script)
    return [
        statement.strip()
        for statement in cleaned.split(";")
        if statement.strip()
    ]


statements = split_sql_statements(sql_script)
print(f"准备执行 SQL 语句数：{len(statements)}")

raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cursor:
        for i, statement in enumerate(statements, start=1):
            cursor.execute(statement)

            if cursor.description is not None:
                cursor.fetchall()

            if i % 10 == 0 or i == len(statements):
                print(f"已执行 {i}/{len(statements)}")

    raw_conn.commit()
    print("DWS + ADS ETL 执行完成。")

except Exception:
    raw_conn.rollback()
    print("ETL 执行失败，已回滚当前事务。")
    raise

finally:
    raw_conn.close()

准备执行 SQL 语句数：40
已执行 10/40
已执行 20/40
已执行 30/40
已执行 40/40
DWS + ADS ETL 执行完成。


In [23]:
# 检查新建的 DWS / ADS 数据表

tables = pd.read_sql(
    """
    SELECT TABLE_NAME
    FROM information_schema.TABLES
    WHERE TABLE_SCHEMA = DATABASE()
      AND (
          LEFT(TABLE_NAME, 4) = 'dws_'
          OR LEFT(TABLE_NAME, 4) = 'ads_'
      )
    ORDER BY TABLE_NAME
    """,
    engine
)

tables

,TABLE_NAME
0,ads_channel_priority
1,ads_core_kpi
2,ads_high_potential_profile
3,ads_high_potential_rule
4,ads_monthly_dashboard
5,dws_monthly_conversion
6,dws_product_depth_conversion
7,dws_product_duration_conversion
8,dws_proxy_funnel
9,dws_session_segment_summary


In [24]:
#DWD -> DWS对账
reconcile_month = pd.read_sql(
    '''
    SELECT
        dwd.dwd_sessions,
        dws.dws_sessions,
        dwd.dwd_purchases,
        dws.dws_purchases,
        dwd.dwd_sessions - dws.dws_sessions AS session_diff,
        dwd.dwd_purchases - dws.dws_purchases AS purchase_diff
    FROM
        (
            SELECT
                COUNT(*) AS dwd_sessions,
                SUM(is_purchase) AS dwd_purchases
            FROM dwd_user_session
        ) dwd
    CROSS JOIN
        (
            SELECT
                SUM(session_cnt) AS dws_sessions,
                SUM(purchase_cnt) AS dws_purchases
            FROM dws_monthly_conversion
        ) dws
    ''',
    engine
)

reconcile_month

,dwd_sessions,dws_sessions,dwd_purchases,dws_purchases,session_diff,purchase_diff
0,12330,12330.0,1908.0,1908.0,0.0,0.0


In [25]:
assert reconcile_month.loc[0, "session_diff"] == 0, "月度 DWS Session 数与 DWD 不一致"
assert reconcile_month.loc[0, "purchase_diff"] == 0, "月度 DWS Purchase 数与 DWD 不一致"

print("月度 DWS 与 DWD 对账通过。")

月度 DWS 与 DWD 对账通过。


In [26]:
#查看月度DWS
monthly_dws = pd.read_sql(
    '''
    SELECT *
    FROM dws_monthly_conversion
    ORDER BY month_num
    ''',
    engine
)

monthly_dws

,month_num,month,session_cnt,purchase_cnt,conversion_rate,avg_product_views,avg_product_duration,avg_bounce_rate,avg_exit_rate,avg_page_value
0,2,Feb,184,3.0,1.63,11.18,471.01,0.0470,0.0741,0.89
1,3,Mar,1907,192.0,10.07,19.81,812.28,0.0217,0.0446,3.96
2,5,May,3364,365.0,10.85,26.49,981.89,0.0269,0.0488,5.43
3,6,June,288,29.0,10.07,36.07,1213.38,0.0351,0.0582,3.39
4,7,Jul,432,66.0,15.28,36.41,1217.60,0.0247,0.0453,4.10
5,8,Aug,433,76.0,17.55,38.26,1272.65,0.0182,0.0377,5.92
6,9,Sep,448,86.0,19.20,33.10,1253.39,0.0122,0.0303,7.56
7,10,Oct,549,115.0,20.95,33.57,1116.98,0.0118,0.0290,8.64
8,11,Nov,2998,760.0,25.35,46.04,1758.40,0.0193,0.0382,7.13
9,12,Dec,1727,216.0,12.51,27.99,1111.47,0.0201,0.0413,6.83


In [27]:
#查看访客类型DWS
visitor_dws = pd.read_sql(
    '''
    SELECT *
    FROM dws_visitor_type_conversion
    ORDER BY conversion_rate DESC
    ''',
    engine
)

visitor_dws

,VisitorType,VisitorType_Name,session_cnt,purchase_cnt,conversion_rate,avg_product_views,avg_product_duration,avg_bounce_rate,avg_exit_rate,avg_page_value
0,New_Visitor,新用户,1694,422.0,24.91,18.05,636.39,0.0053,0.0207,10.77
1,Other,其他,85,16.0,18.82,12.47,570.40,0.0386,0.0633,18.20
2,Returning_Visitor,老用户,10551,1470.0,13.93,34.08,1289.42,0.0248,0.0465,5.00


In [28]:
#渠道：规模和质量一起看DWS
traffic_dws = pd.read_sql(
    '''
    SELECT *
    FROM dws_traffic_type_conversion
    ORDER BY purchase_cnt DESC
    ''',
    engine
)

traffic_dws.head(20)

,TrafficType,session_cnt,purchase_cnt,conversion_rate,purchase_share,avg_product_views,avg_product_duration,avg_page_value
0,2,3913,847.0,21.65,44.39,38.13,1457.94,8.30
1,1,2451,262.0,10.69,13.73,31.92,1234.03,3.45
2,3,2052,180.0,8.77,9.43,25.81,892.76,3.27
3,4,1069,165.0,15.43,8.65,28.53,988.94,7.04
4,8,343,95.0,27.70,4.98,26.12,1084.49,10.31
5,10,450,90.0,20.00,4.72,32.89,1258.31,6.22
6,5,260,56.0,21.54,2.94,17.88,742.33,7.72
7,6,444,53.0,11.94,2.78,29.61,1140.42,5.06
8,20,198,50.0,25.25,2.62,20.15,738.33,15.14
9,11,247,47.0,19.03,2.46,25.21,899.13,5.07


In [29]:
#商品浏览深度DWS
product_depth_dws = pd.read_sql(
    '''
    SELECT *
    FROM dws_product_depth_conversion
    ORDER BY product_view_group
    ''',
    engine
)

product_depth_dws

,product_view_group,session_cnt,purchase_cnt,conversion_rate,avg_product_views,avg_product_duration,avg_page_value
0,01_0-5,2369,102.0,4.31,2.72,97.53,0.91
1,02_6-10,1804,185.0,10.25,7.89,316.49,4.11
2,03_11-20,2560,392.0,15.31,15.12,619.28,6.80
3,04_21-50,3445,685.0,19.88,32.23,1239.74,8.36
4,05_50+,2152,544.0,25.28,102.61,3751.38,7.82


In [30]:
#商品停留时长
duration_dws = pd.read_sql(
    '''
    SELECT *
    FROM dws_product_duration_conversion
    ORDER BY duration_group
    ''',
    engine
)

duration_dws

,duration_group,session_cnt,purchase_cnt,conversion_rate,avg_product_views,avg_product_duration,avg_page_value
0,01_0-2min,2358,94.0,3.99,3.55,40.13,0.59
1,02_2-5min,1808,133.0,7.36,9.94,204.31,2.86
2,03_5-10min,2007,308.0,15.35,16.96,440.73,6.25
3,04_10-20min,2377,482.0,20.28,26.84,864.08,9.23
4,05_20min+,3780,891.0,23.57,70.65,2997.02,8.34


In [31]:
#代理漏斗
proxy_funnel = pd.read_sql(
    '''
    SELECT *
    FROM dws_proxy_funnel
    ORDER BY stage_order
    ''',
    engine
)

proxy_funnel

,stage_order,stage_name,session_cnt,stage_rate,drop_rate
0,1,01_访问网站,12330,100.00,0.00
1,2,02_浏览商品,12292,99.69,0.31
2,3,03_深度浏览,8487,69.04,30.96
3,4,04_最终购买,1908,22.48,77.52


In [32]:
#Session分层DWS
segment_dws = pd.read_sql(
    '''
    SELECT *
    FROM dws_session_segment_summary
    ORDER BY session_cnt DESC
    ''',
    engine
)

segment_dws

,session_segment,session_cnt,purchase_cnt,session_share,avg_product_views,avg_product_duration,avg_bounce_rate,avg_exit_rate,avg_page_value
0,Low_Activity_NonPurchase,8046,0.0,65.26,13.31,517.19,0.0305,0.0549,1.32
1,High_Potential_Rule,2376,0.0,19.27,80.86,2941.96,0.0076,0.0218,4.18
2,Low_Activity_Purchase,1178,1178.0,9.55,18.56,812.32,0.0055,0.0219,31.44
3,High_Value,730,730.0,5.92,96.05,3593.00,0.0045,0.0158,20.54


In [33]:
#进入ADS
core_kpi = pd.read_sql(
    "SELECT * FROM ads_core_kpi",
    engine
)

core_kpi

,total_sessions,purchase_sessions,non_purchase_sessions,conversion_rate,avg_product_views,avg_product_duration,avg_bounce_rate,avg_exit_rate,avg_page_value
0,12330,1908.0,10422.0,15.47,31.73,1194.75,0.0222,0.0431,5.89


In [34]:
monthly_ads = pd.read_sql(
    '''
    SELECT *
    FROM ads_monthly_dashboard
    ORDER BY month_num
    ''',
    engine
)

monthly_ads

,Month_Num,month,session_cnt,purchase_cnt,conversion_rate,avg_product_views,avg_product_duration,avg_bounce_rate,avg_exit_rate,avg_page_value
0,2,Feb,184,3.0,1.63,11.18,471.01,0.0470,0.0741,0.89
1,3,Mar,1907,192.0,10.07,19.81,812.28,0.0217,0.0446,3.96
2,5,May,3364,365.0,10.85,26.49,981.89,0.0269,0.0488,5.43
3,6,June,288,29.0,10.07,36.07,1213.38,0.0351,0.0582,3.39
4,7,Jul,432,66.0,15.28,36.41,1217.60,0.0247,0.0453,4.10
5,8,Aug,433,76.0,17.55,38.26,1272.65,0.0182,0.0377,5.92
6,9,Sep,448,86.0,19.20,33.10,1253.39,0.0122,0.0303,7.56
7,10,Oct,549,115.0,20.95,33.57,1116.98,0.0118,0.0290,8.64
8,11,Nov,2998,760.0,25.35,46.04,1758.40,0.0193,0.0382,7.13
9,12,Dec,1727,216.0,12.51,27.99,1111.47,0.0201,0.0413,6.83


In [35]:
channel_ads = pd.read_sql(
    '''
    SELECT *
    FROM ads_channel_priority
    ORDER BY purchase_volume_rank, cvr_rank
    ''',
    engine
)

channel_ads.head(20)

,TrafficType,session_cnt,purchase_cnt,conversion_rate,purchase_share,avg_product_views,avg_product_duration,avg_page_value,purchase_volume_rank,cvr_rank
0,2,3913,847.0,21.65,44.39,38.13,1457.94,8.30,1,5
1,1,2451,262.0,10.69,13.73,31.92,1234.03,3.45,2,12
2,3,2052,180.0,8.77,9.43,25.81,892.76,3.27,3,14
3,4,1069,165.0,15.43,8.65,28.53,988.94,7.04,4,9
4,8,343,95.0,27.70,4.98,26.12,1084.49,10.31,5,3
5,10,450,90.0,20.00,4.72,32.89,1258.31,6.22,6,7
6,5,260,56.0,21.54,2.94,17.88,742.33,7.72,7,6
7,6,444,53.0,11.94,2.78,29.61,1140.42,5.06,8,11
8,20,198,50.0,25.25,2.62,20.15,738.33,15.14,9,4
9,11,247,47.0,19.03,2.46,25.21,899.13,5.07,10,8


In [42]:
#高潜Session ADS
high_potential_ads = pd.read_sql(
    '''
    SELECT *
    FROM ads_high_potential_rule
    ORDER BY ProductRelated DESC
    LIMIT 20
    ''',
    engine
)

high_potential_ads

,session_id,Month_Num,month,VisitorType,VisitorType_Name,TrafficType,is_weekend,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,TotalPage_Views,TotalPage_Duration,ProductView_Ratio,is_Purchase,rule_name
0,5153,5,May,Returning_Visitor,老用户,14,1,705,43171.233380,0.004851,0.015431,1.0,746,47850.233380,0.945040,0,P75_ProductRelated>=38
1,6685,8,Aug,Returning_Visitor,老用户,1,0,686,23342.082050,0.009853,0.022771,0.0,713,23840.082050,0.962132,0,P75_ProductRelated>=38
2,8309,11,Nov,Returning_Visitor,老用户,8,0,584,24844.156200,0.002099,0.009347,5.0,613,25825.156200,0.952692,0,P75_ProductRelated>=38
3,6788,10,Oct,Returning_Visitor,老用户,2,0,518,11976.721350,0.000038,0.003837,0.0,526,12138.721350,0.984791,0,P75_ProductRelated>=38
4,6166,7,Jul,Returning_Visitor,老用户,3,0,486,23050.104140,0.000324,0.011249,0.0,499,25891.104140,0.973948,0,P75_ProductRelated>=38
5,8072,12,Dec,Returning_Visitor,老用户,2,0,449,63973.522230,0.000764,0.027701,0.0,460,69921.522230,0.976087,0,P75_ProductRelated>=38
6,4675,5,May,Returning_Visitor,老用户,20,1,440,9951.869139,0.001850,0.010098,2.0,451,10033.869139,0.975610,0,P75_ProductRelated>=38
7,5917,9,Sep,Returning_Visitor,老用户,13,0,439,21857.046480,0.003589,0.012498,11.0,455,23615.046480,0.964835,0,P75_ProductRelated>=38
8,6675,9,Sep,Returning_Visitor,老用户,1,0,429,9661.585763,0.003044,0.012547,3.0,449,10054.585763,0.955457,0,P75_ProductRelated>=38
9,9902,11,Nov,Returning_Visitor,老用户,1,1,423,17086.234240,0.006294,0.021599,0.0,429,17164.234240,0.986014,0,P75_ProductRelated>=38


In [43]:
high_potential_profile = pd.read_sql(
    "SELECT * FROM ads_high_potential_profile",
    engine
)

high_potential_profile

,high_potential_sessions,share_of_non_purchase,avg_product_views,avg_product_duration,avg_bounce_rate,avg_exit_rate,avg_page_value
0,2376,22.8,80.86,2941.96,0.0076,0.0218,4.18


In [44]:
#DWS -> ADS再次对账
reconcile_ads = pd.read_sql(
    '''
    SELECT
        dws.dws_sessions,
        ads.ads_sessions,
        dws.dws_purchases,
        ads.ads_purchases,
        dws.dws_sessions - ads.ads_sessions AS session_diff,
        dws.dws_purchases - ads.ads_purchases AS purchase_diff
    FROM
        (
            SELECT
                SUM(session_cnt) AS dws_sessions,
                SUM(purchase_cnt) AS dws_purchases
            FROM dws_monthly_conversion
        ) dws
    CROSS JOIN
        (
            SELECT
                SUM(session_cnt) AS ads_sessions,
                SUM(purchase_cnt) AS ads_purchases
            FROM ads_monthly_dashboard
        ) ads
    ''',
    engine
)

reconcile_ads

,dws_sessions,ads_sessions,dws_purchases,ads_purchases,session_diff,purchase_diff
0,12330.0,12330.0,1908.0,1908.0,0.0,0.0


In [45]:
assert reconcile_ads.loc[0, "session_diff"] == 0, "ADS 月度 Session 数与 DWS 不一致"
assert reconcile_ads.loc[0, "purchase_diff"] == 0, "ADS 月度 Purchase 数与 DWS 不一致"

print("DWS → ADS 月度指标对账通过。")

DWS → ADS 月度指标对账通过。


In [46]:
#统一数据质量检查

quality_checks = {}

quality_checks["dwd_not_empty"] = int(dwd_check.loc[0, "dwd_rows"]) > 0
quality_checks["monthly_reconcile_session"] = reconcile_month.loc[0, "session_diff"] == 0
quality_checks["monthly_reconcile_purchase"] = reconcile_month.loc[0, "purchase_diff"] == 0
quality_checks["ads_reconcile_session"] = reconcile_ads.loc[0, "session_diff"] == 0
quality_checks["ads_reconcile_purchase"] = reconcile_ads.loc[0, "purchase_diff"] == 0
quality_checks["monthly_cvr_range"] = monthly_dws["conversion_rate"].between(0, 100).all()
quality_checks["monthly_purchase_le_session"] = (
    (monthly_dws["purchase_cnt"] <= monthly_dws["session_cnt"]).all()
)

quality_result = pd.DataFrame({
    "check_name": quality_checks.keys(),
    "passed": quality_checks.values()
})

quality_result

,check_name,passed
0,dwd_not_empty,True
1,monthly_reconcile_session,True
2,monthly_reconcile_purchase,True
3,ads_reconcile_session,True
4,ads_reconcile_purchase,True
5,monthly_cvr_range,True
6,monthly_purchase_le_session,True


In [48]:
failed_checks = quality_result.loc[
    quality_result["passed"] == False,
    "check_name"
].tolist()

if failed_checks:
    raise AssertionError(
        "以下数据质量检查未通过：\n"
        + "\n".join(failed_checks)
    )

print("基础数据质量检查全部通过。")

基础数据质量检查全部通过。
